# NB03a — Synthetic Generation (LoRA-Boost)

สร้าง synthetic image สำหรับ condition D ของ NB04 โหลด per-species LoRA จาก NB02 มาสวมบน FLUX.2-klein แล้ว gen ภาพที่ res 512 ตามสัดส่วน organ จริงของ species นั้น แบ่ง budget ต่อ species เท่ากับ need = T - n_train(real) เพื่อให้ NB04 fill ถึง T พอดี

output: data/metadata/synthetic_manifest.csv (path, species_id, source=lora_boost) ที่ NB04 รออยู่

## Section 1) Setup & load handoff

ตั้ง path โหลด nb02_handoff.json + lora_registry.csv จาก NB02 ผูก absolute path ของ LoRA weights แต่ละตัวแล้ว validate ว่าครบ 15 ก่อนไปต่อ source ล็อกเป็น lora_boost (03a = condition D)

In [13]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('/workspace/lora-boost')
DATA = ROOT / 'data'
META = DATA / 'metadata'
IMG_ROOT = DATA / 'raw' / 'plantnet_300K' / 'images'
SYNTH_ROOT = DATA / 'synthetic'
RESULTS = ROOT / 'results' / 'nb03'

SOURCE = 'lora_boost'                       # 03a = condition D
SYNTH_DIR = SYNTH_ROOT / SOURCE             # curated สุดท้ายเก็บที่ data/synthetic/lora_boost/<id>/
for p in (SYNTH_DIR, RESULTS):
    p.mkdir(parents=True, exist_ok=True)

handoff = json.load(open(META / 'nb02_handoff.json'))
reg_df = pd.read_csv(ROOT / handoff['registry'], dtype={'species_id': str})
reg_df['weights_abs'] = reg_df['weights_path'].map(lambda p: ROOT / p)
reg_df['weights_ok'] = reg_df['weights_abs'].map(lambda p: p.exists())

BASE_MODEL = handoff['base_model']
CAPTION_TMPL = handoff['caption_tmpl']      # 'a photo of the {organ} of a {trigger} plant'

print(f'source      :\t{SOURCE}')
print(f'base model  :\t{BASE_MODEL}')
print(f'caption     :\t{CAPTION_TMPL}')
print(f'gen (nb02)  :\t{handoff["gen_defaults"]}')
print(f'species     :\t{len(reg_df)}')
print(f'weights ok  :\t{int(reg_df.weights_ok.sum())} / {len(reg_df)}')
print(f'flagged     :\t{reg_df.loc[reg_df.flag != "ok", "species_id"].tolist()}')

source      :	lora_boost
base model  :	black-forest-labs/FLUX.2-klein-base-4B
caption     :	a photo of the {organ} of a {trigger} plant
gen (nb02)  :	{'steps': 28, 'guidance': 4.0, 'res': 1024}
species     :	15
weights ok  :	15 / 15
flagged     :	['1406486', '1406863']


## Section 2) Config & generation plan

รวมค่า gen ไว้ใน cfg ที่เดียว แล้ววางแผนต่อ species จาก splits.csv: budget = need = T - n_train(real) เพื่อให้ NB04 fill ถึง T พอดี และแบ่ง budget เป็น flower/leaf ตามสัดส่วน organ จริงใน train ของ species นั้น

In [14]:
import numpy as np

cfg = {
    'resolution': 512,                  # downstream classifier resize เหลือ 224 อยู่แล้ว 512 พอเหลือ
    'steps': 28,
    'guidance': 4.0,
    'seed': 42,
    'gen_batch': 1,                
    'T': 200,                           # fill target ต่อ rare species (ต้องตรงกับ NB04)
}

for k, v in cfg.items():
    print(f'{k:20}:\t{v}')

resolution          :	512
steps               :	28
guidance            :	4.0
seed                :	42
gen_batch           :	1
T                   :	200


In [15]:
import numpy as np

splits_df = pd.read_csv(META / 'splits.csv', dtype={'species_id': str})
rare_train = splits_df[(splits_df.is_rare) & (splits_df.split == 'train')]

rows = []
for r in reg_df.itertuples():
    sid = r.species_id
    g = rare_train[rare_train.species_id == sid]
    n_flower = int((g.organ == 'flower').sum())
    n_leaf = int((g.organ == 'leaf').sum())
    n_train = n_flower + n_leaf
    budget = max(0, cfg['T'] - n_train)
    fl_frac = n_flower / n_train if n_train else 0.0
    gen_flower = round(budget * fl_frac)
    gen_leaf = budget - gen_flower
    rows.append({
        'species_id': sid,
        'name': ' '.join(r.scientific_name.split()[:2]),
        'n_train': n_train,
        'budget': budget,
        'gen_flower': gen_flower,
        'gen_leaf': gen_leaf,
    })

plan_df = pd.DataFrame(rows).sort_values('budget', ascending=False).reset_index(drop=True)

print(plan_df.to_string(index=False))
print(f'\ntotal budget :\t{plan_df.budget.sum():,} images')

species_id                         name  n_train  budget  gen_flower  gen_leaf
   1416509        Nepenthes vieillardii       20     180          72       108
   1419807           Vanilla planifolia       23     177          46       131
   1376703           Petiveria alliacea       28     172          43       129
   1409195            Pereskia aculeata       33     167          40       127
   1408557             Pongamia pinnata       36     164          50       114
   1406486             Kigelia africana       37     163          70        93
   1420787               Alocasia odora       38     162          30       132
   1408490         Pereskia grandifolia       40     160         116        44
   1385700        Barringtonia asiatica       47     153          85        68
   1356076         Tradescantia zebrina       49     151          22       129
   1397598              Cobaea scandens       49     151         117        34
   1618661 Dendrobium victoriae-reginae       51    

## Section 3) Generation

โหลด FLUX.2-klein ครั้งเดียวแล้วสวม LoRA ทีละ species generate ตาม budget ใน plan_df แยก flower/leaf ตามแผน seed ไล่ไม่ซ้ำเพื่อความหลากหลาย เซฟรูปลง data/synthetic/lora_boost/<id>/ พร้อมจด gen_manifest (species_id, organ, seed, path) ทุกใบ idempotent: species ไหนมีครบ budget แล้วข้าม pod หลุดมารันต่อได้

In [16]:
import torch
from diffusers import Flux2KleinPipeline

if 'pipe' not in dir():
    pipe = Flux2KleinPipeline.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16).to('cuda')
print(f'pipe loaded :\t{torch.cuda.memory_allocated() / 1e9:.1f} GB')

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

pipe loaded :	16.4 GB


In [17]:
from pathlib import Path

GEN_MANIFEST = RESULTS / 'gen_manifest.csv'
GEN_COLS = ['species_id', 'organ', 'seed', 'path']

def load_gen_manifest():
    if GEN_MANIFEST.exists():
        return pd.read_csv(GEN_MANIFEST, dtype={'species_id': str, 'seed': int})
    return pd.DataFrame(columns=GEN_COLS)

@torch.no_grad()
def generate_species(sid, trigger, gen_flower, gen_leaf, done_seeds):
    out_dir = SYNTH_DIR / sid
    out_dir.mkdir(parents=True, exist_ok=True)
    pipe.load_lora_weights(str(ROOT / reg_df.loc[reg_df.species_id == sid, 'weights_path'].iloc[0]))

    rows = []
    seed = 0
    for organ, n_target in (('flower', gen_flower), ('leaf', gen_leaf)):
        prompt = CAPTION_TMPL.format(organ=organ, trigger=trigger)
        made = 0
        while made < n_target:
            bs = min(cfg['gen_batch'], n_target - made)
            seeds = []
            while len(seeds) < bs:
                if (sid, seed) not in done_seeds:
                    seeds.append(seed)
                seed += 1
            gens = [torch.Generator('cuda').manual_seed(s) for s in seeds]
            imgs = pipe(prompt=prompt, num_inference_steps=cfg['steps'],
                        guidance_scale=cfg['guidance'],
                        height=cfg['resolution'], width=cfg['resolution'],
                        num_images_per_prompt=bs, generator=gens).images
            for s, img in zip(seeds, imgs):
                fn = f'{sid}_{organ}_{s:05d}.png'
                img.save(out_dir / fn)
                rows.append({'species_id': sid, 'organ': organ, 'seed': s, 'path': fn})
            made += bs
    pipe.unload_lora_weights()
    return rows

In [18]:
import time

for r in plan_df.itertuples():
    sid = r.species_id
    trigger = reg_df.loc[reg_df.species_id == sid, 'trigger'].iloc[0]

    man = load_gen_manifest()
    have = man[man.species_id == sid]
    if len(have) >= r.budget:
        print(f'{sid} {r.name:28} skip (have {len(have)}/{r.budget})')
        continue

    done_seeds = set(zip(have.species_id, have.seed))
    t0 = time.time()
    rows = generate_species(sid, trigger, r.gen_flower, r.gen_leaf, done_seeds)

    out = pd.concat([man, pd.DataFrame(rows)], ignore_index=True)
    out.to_csv(GEN_MANIFEST, index=False)
    print(f'{sid} {r.name:28} +{len(rows):4d} -> {len(out[out.species_id == sid]):4d}/{r.budget}  {(time.time() - t0) / 60:.1f} min')

man = load_gen_manifest()
print(f'\ntotal generated :\t{len(man):,} / {plan_df.budget.sum():,}')

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

...

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

1392654 Gomphocarpus physocarpus     + 129 ->  129/129  14.0 min

total generated :	2,356 / 2,356


## Section 4) Finalize

เขียน synthetic_manifest.csv จาก gen_manifest ใช้รูปทั้งหมดที่ generate มา ไม่กรอง path เก็บเป็น <species_id>/<filename> ให้ NB04 เติม prefix ของ source เองตอน fill rare coverage report ปิดท้ายเพื่อยืนยันว่าครบ budget ทุก species

In [19]:
man = pd.read_csv(GEN_MANIFEST, dtype={'species_id': str})

out = pd.DataFrame({
    'path': man.species_id + '/' + man.path,
    'species_id': man.species_id,
    'source': SOURCE,
})

SYNTH_MANIFEST = META / 'synthetic_manifest.csv'
out.to_csv(SYNTH_MANIFEST, index=False)

cov = out.groupby('species_id').size()
plan_df['n_synth'] = plan_df.species_id.map(cov).fillna(0).astype(int)
plan_df['ok'] = plan_df.n_synth >= plan_df.budget

print(plan_df[['species_id', 'name', 'n_train', 'budget', 'n_synth', 'ok']].to_string(index=False))
print(f'\nmanifest :\t{SYNTH_MANIFEST}')
print(f'rows     :\t{len(out):,}')
print(f'all ok   :\t{plan_df.ok.all()}')

species_id                         name  n_train  budget  n_synth   ok
   1416509        Nepenthes vieillardii       20     180      180 True
   1419807           Vanilla planifolia       23     177      177 True
   1376703           Petiveria alliacea       28     172      172 True
   1409195            Pereskia aculeata       33     167      167 True
   1408557             Pongamia pinnata       36     164      164 True
   1406486             Kigelia africana       37     163      163 True
   1420787               Alocasia odora       38     162      162 True
   1408490         Pereskia grandifolia       40     160      160 True
   1385700        Barringtonia asiatica       47     153      153 True
   1356076         Tradescantia zebrina       49     151      151 True
   1397598              Cobaea scandens       49     151      151 True
   1618661 Dendrobium victoriae-reginae       51     149      149 True
   1384499          Lagenaria siceraria       56     144      144 True
   140

In [20]:
print(out.head(3).to_string(index=False))
print()
sample = out.iloc[0]
full = SYNTH_ROOT / sample.source / sample.path
print(f'manifest path  :\t{sample.path}')
print(f'resolves to    :\t{full}')
print(f'file exists    :\t{full.exists()}')

                            path species_id     source
1416509/1416509_flower_00000.png    1416509 lora_boost
1416509/1416509_flower_00001.png    1416509 lora_boost
1416509/1416509_flower_00002.png    1416509 lora_boost

manifest path  :	1416509/1416509_flower_00000.png
resolves to    :	/workspace/lora-boost/data/synthetic/lora_boost/1416509/1416509_flower_00000.png
file exists    :	True
